
$$
\begin{array}{c}
{\Huge \text{CAUSAL INFERENCE LAB 3}}\\\\
{\Large \textbf{Abhishek Bakshi}} \\
{\large \textit{Center for Data Science, New York University}} \\\\
{\Large \text{Office Hours: Wednesdays, 12:30 - 1:30 pm}} \\
{\Large \textit{Room 242, 60 5th Ave, 2nd floor}} \\\\
\text{Materials prepared by: Abhishek Bakshi, Xiang Gao}
\end{array}
$$

---



<p align="center">
  <a href="https://xkcd.com/2118/">
    <img src="https://imgs.xkcd.com/comics/normal_distribution.png"
         alt="xkcd: Normal Distribution" width="760">
  </a>
</p>



<h1 style="font-size: 42px;">Statistics</h1>

---



<h1 style="font-size: 30px;">Today's Goals:</h1>

<ol style="font-size: 20px; line-height: 1.6;">
  <li>Parameter vs estimator vs estimate</li>
  <li>Sampling distributions, bias, variance, and consistency</li>
  <li>WLLN and CLT</li>
  <li>Hypothesis tests, p-value, and significance level</li>
  <li>Type I and Type II errors</li>
</ol>

---



<style>
body,
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.markdown-body,
.vscode-body {
  font-family: Arial, Helvetica, sans-serif !important;
}

p, li, blockquote, td, th {
  font-size: 20px !important;
  line-height: 1.6 !important;
}

h1 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 42px !important;
  line-height: 1.25 !important;
  margin-top: 0.6em !important;
}

h2 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 30px !important;
  line-height: 1.3 !important;
  margin-top: 0.7em !important;
  border-bottom: 2px solid #4b4bb7;
  padding-bottom: 0.15em;
}

h3 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 25px !important;
  line-height: 1.35 !important;
  margin-top: 0.7em !important;
}

h4 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 22px !important;
  line-height: 1.4 !important;
}

table {
  font-size: 19px !important;
  line-height: 1.45 !important;
}

code, pre {
  font-size: 17px !important;
}

blockquote {
  border-left: 5px solid #4b4bb7 !important;
  padding: 0.3em 0.8em !important;
  color: inherit !important;
}

.MathJax, mjx-container {
  font-size: 115% !important;
}
</style>


In [ ]:

# If an import fails, run this in a separate cell:
# %pip install numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.precision", 4)
plt.rcParams.update({
    "font.size": 15, "axes.titlesize": 19, "axes.labelsize": 16,
    "xtick.labelsize": 13, "ytick.labelsize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

rng = np.random.default_rng(2026)
print("Ready for statistics: samples, estimators, and inference!")



## 1. Parameters, Estimates and Estimators

---

### Probability and statistics

**Probability:** start with a known distribution and derive properties of random variables drawn from it.

**Statistics:** start with observed data and use it to learn about an unknown distribution or one of its parameters.

Suppose $X_1,\ldots,X_n$ are independent and identically distributed (i.i.d.) draws from a population distribution ($X$).

- A **parameter** is a fixed feature of the population, such as $\mu=\mathbb E[X]$.
- An **estimator** is a rule that maps a random sample to a number, such as $\bar X_n=\frac{1}{n}\sum_{i=1}^n X_i$.
- An **estimate** is the numerical value produced after we observe a particular sample.

> We will expose the data-generating process in this lab so that we can check our work. In real data, the population parameter is unknown.


In [ ]:
# Simulated population of waits for the next campus shuttle (minutes).
wait_population = rng.exponential(scale=10, size=200_000)
wait_mean = wait_population.mean()
wait_sd = wait_population.std(ddof=0)

print(f"Toy population mean wait: {wait_mean:.2f} minutes")
print(f"Toy population SD:        {wait_sd:.2f} minutes")

plt.figure(figsize=(9, 5))

plt.hist(
    wait_population,
    bins=100,
    density=True,
    alpha=0.6,
    edgecolor="black"
)

# Theoretical exponential density
x = np.linspace(0, wait_population.max(), 1000)
pdf = (1 / 10) * np.exp(-x / 10)

plt.plot(x, pdf, linewidth=2, label="Theoretical Exp(scale=10)")

# Mean
plt.axvline(
    wait_mean,
    linestyle="--",
    linewidth=2,
    label=f"Mean = {wait_mean:.2f}"
)

plt.xlabel("Wait time")
plt.ylabel("Density")
plt.title("Wait-Time Population Distribution")
plt.legend()
plt.show()


### Question 1: samples

Draw three independent samples of $n=30$ shuttle waits from the same population.

1. Which object is the parameter?
2. Will the three sample means be identical? Should they be near the population mean?


In [ ]:

# Three random samples and their estimates of the same wait_population mean
n_small = 30
three_samples = rng.choice(wait_population, size=(3, n_small), replace=True)
three_means = three_samples.mean(axis=1)

display(pd.DataFrame({
    "Sample": ["A", "B", "C"],
    "Sample size": n_small,
    "Sample mean (estimate)": three_means,
}))

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, sample, estimate, label in zip(axes, three_samples, three_means, ["A", "B", "C"]):
    ax.scatter(sample, np.zeros_like(sample), color="#6554B1", alpha=0.75)
    ax.axvline(wait_mean, color="#2D2A4A", ls="--", lw=2, label="Population mean wait")
    ax.axvline(estimate, color="#59B997", lw=2, label="Sample mean")
    ax.set(title=f"Sample {label}: same wait_population, different estimate", yticks=[])
axes[-1].set(xlabel="Shuttle wait (minutes)")
axes[0].legend(loc="upper right", fontsize=11)
plt.show()



### Sampling distributions

Because $\bar X_n$ depends on the random sample, it is itself a random variable. Its distribution over repeated samples is the **sampling distribution** of the sample mean.

For i.i.d. draws with population mean $\mu$ and variance $\sigma^2$,

$$
\mathbb E[\bar X_n]=\mu,
\qquad
\operatorname{Var}(\bar X_n)=\frac{\sigma^2}{n},
\qquad
\operatorname{SE}(\bar X_n)=\frac{\sigma}{\sqrt n}.
$$


### Question 2: bias and spread

Consider two estimators of the population mean:

$$
\hat\mu_A=\bar X_n,
\qquad
\hat\mu_B=\frac{n-1}{n}\bar X_n.
$$

1. Calculate $\mathbb E[\hat\mu_A]$ and $\mathbb E[\hat\mu_B]$. Which estimator is unbiased?
2. If $\mu=10$, what is the bias of $\hat\mu_B$ when $n=25$? When $n=100$?
3. Which sample mean, $\bar X_{25}$ or $\bar X_{100}$, has the smaller standard error? By what factor?
4. $\hat\mu_B$ has less spread than $\hat\mu_A$, but is shifted downward. What is the trade-off?

In [ ]:
# Repeated sampling makes bias and variability visible
n_rep, n_a, n_b = 4_000, 25, 100

means_a = rng.choice(wait_population, size=(n_rep, n_a), replace=True).mean(axis=1)
means_b = rng.choice(wait_population, size=(n_rep, n_b), replace=True).mean(axis=1)

shrunken_a = (n_a - 1) / n_a * means_a
shrunken_b = (n_b - 1) / n_b * means_b

summary = pd.DataFrame({
    "Estimator": [
        f"X-bar (n={n_a})",
        f"(n - 1) / n × X-bar (n={n_a})",
        f"X-bar (n={n_b})",
        f"(n - 1) / n × X-bar (n={n_b})",
    ],
    "Average across repeated samples": [
        means_a.mean(),
        shrunken_a.mean(),
        means_b.mean(),
        shrunken_b.mean(),
    ],
    "Estimated bias": [
        means_a.mean() - wait_mean,
        shrunken_a.mean() - wait_mean,
        means_b.mean() - wait_mean,
        shrunken_b.mean() - wait_mean,
    ],
    "Empirical SD of estimates": [
        means_a.std(ddof=0),
        shrunken_a.std(ddof=0),
        means_b.std(ddof=0),
        shrunken_b.std(ddof=0),
    ],
    "Theoretical SE": [
        wait_sd / np.sqrt(n_a),
        (n_a - 1) / n_a * wait_sd / np.sqrt(n_a),
        wait_sd / np.sqrt(n_b),
        (n_b - 1) / n_b * wait_sd / np.sqrt(n_b),
    ],
})

display(summary)
print(f"SE(n=25) / SE(n=100) = {np.sqrt(n_b / n_a):.1f}")


## 2. Large samples

---

### Weak Law of Large Numbers

An estimator ($\hat{\theta}_n$) is **consistent** for a population parameter $\theta$ if, for every $\varepsilon>0$,

$$
P\left(\left|\hat{\theta}_n-\theta\right|>\varepsilon\right)\to 0
\quad\text{as }n\to\infty.
$$

Under the usual i.i.d. and finite-variance conditions, the WLLN says that $\bar X_n$ is consistent for $\mu$.


### Question 3: a growing sample

As we observe more shuttle waits from this same population

- Does the sample mean have to move closer to $\mu$ at every step?
- By WLLN, is $\hat{\mu}_b$ a consistent estimator of $\mu$?


In [ ]:
# One growing i.i.d. sample: a visual for consistency
long_sample = rng.choice(wait_population, size=10_000, replace=True)

n_values = np.arange(1, len(long_sample) + 1)
running_mean = np.cumsum(long_sample) / n_values
running_mu_b = (n_values - 1) / n_values * running_mean

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(
    n_values,
    running_mean,
    color="#6554B1",
    lw=1.5,
    label=r"$\bar X_n$",
)

ax.plot(
    n_values,
    running_mu_b,
    color="#FF0A0A",
    lw=1.5,
    alpha=0.4,
    label=r"$\hat\mu_{B,n}=\frac{n-1}{n}\bar X_n$",
)

ax.axhline(
    wait_mean,
    color="#CF4D4D",
    ls="--",
    lw=2,
    label="Population mean wait",
)

ax.set(
    title="WLLN: both estimators settle near the population mean",
    xlabel="Number of observations (log scale)",
    ylabel="Estimated mean wait (minutes)",
    xlim=(1, len(long_sample)),
)
ax.set_xscale("log")

ax.legend(fontsize=11)
plt.show()


### Central Limit Theorem

The WLLN says that the sample mean gets close to $\mu$. The CLT describes the **shape** of its sampling error:

$$
Z = \frac{\sqrt n(\bar X_n-\mu)}{\sigma}
\overset{d}{\longrightarrow}N(0,1).
$$

For large $n$, this means approximately

$$
\bar X_n\approx N\left(\mu,\frac{\sigma^2}{n}\right).
$$

The population itself need not be normal, it can have any distribution - continuous or discrete (with finite variance).



### Question 4: Sampling distribution of the sample mean

What is the approximate large sample distribution of the sample mean from the shuttle wait-time example?


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Sampling distributions from a non-normal wait population
n_values = [1, 2, 3, 5, 8, 12, 20, 30, 50, 75, 100]
B_clt = 3_000
max_n = max(n_values)

# Each row is one growing i.i.d. sample.
draws = rng.choice(wait_population, size=(B_clt, max_n), replace=True)
cumulative_sums = np.cumsum(draws, axis=1)

grid = np.linspace(-4, 4, 300)
bins = np.linspace(-4, 4, 29)

fig, ax = plt.subplots(figsize=(10, 5))

def update(frame):
    ax.clear()

    n_current = n_values[frame]
    sample_means = cumulative_sums[:, n_current - 1] / n_current
    z_scores = np.sqrt(n_current) * (sample_means - 10) / 10

    ax.hist(
        z_scores,
        bins=bins,
        density=True,
        color="#C7C3E8",
        edgecolor="white",
        label="Standardized sample means",
    )
    ax.plot(
        grid,
        stats.norm.pdf(grid),
        color="#CF4D4D",
        lw=2,
        label="Standard normal",
    )

    ax.set(
        title=f"n = {n_current}",
        xlabel="Z-score",
        ylabel="Density",
        xlim=(-4, 4),
    )
    ax.legend(fontsize=11)

anim = FuncAnimation(
    fig,
    update,
    frames=len(n_values),
    interval=400,
    repeat=True,
)

plt.close(fig)
display(HTML(anim.to_jshtml()))

## 3. Hypothesis tests

---

### A two-sided test

Suppose a transportation club records one random sample of 100 shuttle wait times and wants to assess the claim that the population mean wait is 10 minutes:

$$
H_0:\mu=10,
\qquad
H_1:\mu\ne10.
$$

We use the large-sample standardized statistic

$$
Z=\frac{\bar X-\mu_0}{s/\sqrt n}.
$$

Under $H_0$, the CLT implies that, for a large i.i.d. sample with finite variance,

$$
Z\overset{\text{approx.}}{\sim}N(0,1).
$$

This is an approximation: individual shuttle waits are exponential and right-skewed, so \(Z\) is not exactly normal in a finite sample.

In [ ]:
# A new hypothetical sample of shuttle waits
observed_waits = rng.exponential(scale=15, size=100)

mu_0 = 10
alpha = 0.05

n_obs = len(observed_waits)
observed_mean = observed_waits.mean()
observed_sd = observed_waits.std(ddof=1)

print(f"Sample size n = {n_obs}")
print(f"Sample mean wait = {observed_mean:.2f} minutes")
print(f"Sample SD = {observed_sd:.2f} minutes")

### Question 5: test the claim

For this sample, test $H_0:\mu=10$ against $H_1:\mu\ne10$.

1. Is the sample mean above or below 10?
2. Calculate the signed standardized statistic $Z$.
3. Using the standard normal distribution, is the two-sided p-value smaller than 0.05? What is the decision at $\alpha=0.05$?

In [ ]:
# Two-sided large-sample normal approximation
standard_error = observed_sd / np.sqrt(n_obs)
z_statistic = (observed_mean - mu_0) / standard_error

# P(|N(0, 1)| >= |z|)
p_value = 2 * stats.norm.cdf(-abs(z_statistic))

print(f"Z = {z_statistic:.3f}")
print(f"Two-sided p-value = {p_value:.5f}")
print(
    f"Decision at alpha = {alpha}: "
    f"{'reject H0' if p_value < alpha else 'fail to reject H0'}"
)

### P-values and errors

The **p-value** is the probability, **assuming $H_0$ is true**, of observing a test statistic at least as extreme as the one observed. It is **not** the probability that $H_0$ is true.

- A smaller p-value is stronger evidence against $H_0$.
- Choose the significance level $\alpha$ before looking at the data. Reject $H_0$ when $p<\alpha$; otherwise, **fail to reject** $H_0$. Common $\alpha$ values are 0.05, 0.01, 0.001.
- Failing to reject $H_0$ does not show that it is true. The sample may be too small or too noisy to detect a difference.

| What is actually true? | Reject $H_0$ | Fail to reject $H_0$ |
|---|---|---|
| $H_0$ is true | **Type I error** | Correct decision |
| $H_0$ is false | Correct decision | **Type II error** |

A Type I error has long-run probability at most $\alpha$ when the test is correctly specified. A Type II error is failing to reject a false null. We will not calculate Type II error probabilities today, but keep the distinction in mind.

### Question 6: Court trials

Take the scenario where a person is brought in front of a judge for trial.

1. Give an example of Type I error in this scenario.
2. Give an example of Type II error in this scenario.